In [4]:
import pandas as pd
from scipy.stats import false_discovery_control

# -----------------------------
# File paths
# -----------------------------
input_file = "Raw-p-val.xlsx"
output_file = "Raw-p-val_FDR_corrected.xlsx"

# -----------------------------
# Read all sheets
# -----------------------------
xls = pd.ExcelFile(input_file)
sheet_names = xls.sheet_names

# Dictionary to store corrected sheets
corrected_sheets = {}

# -----------------------------
# Loop through each sheet
# -----------------------------
for sheet in sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet)

    # Make a copy so original stays unchanged
    df = df.copy()

    # Make sure p-values are numeric
    df["p-value"] = pd.to_numeric(df["p-value"], errors="coerce")

    # Find rows where p-values are present
    mask = df["p-value"].notna()

    # Apply Benjamini-Hochberg FDR correction only to non-missing p-values
    df.loc[mask, "FDR_BH_pvalue"] = false_discovery_control(
        df.loc[mask, "p-value"].to_numpy(),
        method="bh"
    )

    # Optional: add a significance column at FDR < 0.05
    df["FDR_BH_significant_0.05"] = df["FDR_BH_pvalue"] < 0.05

    corrected_sheets[sheet] = df

# -----------------------------
# Write all corrected sheets to a new Excel file
# -----------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet, df in corrected_sheets.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

print(f"Done. New file saved as: {output_file}")

Done. New file saved as: Raw-p-val_FDR_corrected.xlsx
